# Exercise 7 : Preference Learning and Reward Modelling for Language Models

### Goals

1. Learn about pairwise preferences in language modeling
2. Learn about the Bradley Terry and Plackett-Luce model
3. Collecting human vs. AI feedback
4. Calibration and Label quality
5. Reward Hacking





## Background - Why Do We Need Reward Models in Language?

A central component of reinforcement learning is the reward model, which tells the agent whether its behaviour is desirable. In domains such as chess or Go, reward models are easy to define: it is trivial to determine whether a player has won or lost. This simplicity is a key reason why superhuman performance has been achieved in these settings.

Language, however, is fundamentally different. It is highly flexible and ambiguous, and there are countless ways to express the same idea. Writing a handcrafted reward function that correctly evaluates all acceptable (and unacceptable) language outputs is effectively impossible.

## Example: A Hand-Crafted Insult Reward Model

Problem:  
Our model keeps insulting users, even when they ask harmless questions.

Solution:  
We decide to fine-tune the model using reinforcement learning, but first we need a reward model that penalizes insulting behaviour.

A naïve approach might be to define a reward function that scans the model’s output for a list of offensive or “bad” words, and assigns a negative reward if any of them appear. This approach can catch some insults, but it quickly breaks down.

For example:
- It is easy to construct insults that do not use any explicitly offensive words.
- The model might need to mention offensive words in a non-insulting context, such as explaining why a particular term is hurtful.

In both cases, the reward model fails to capture what we actually care about. It does not understand intent, context, or nuance—it only checks for surface-level patterns. As a result, it cannot reliably distinguish between harmful and harmless language.

## Reinforcement Learning from Human Feedback - RLHF

A solution to this problem was inspired by earlier work in reinforcement learning. Christiano et al. (2017) posed a similar challenge: how do you write a reward function that recognizes a backflip in robotics? Humans can easily identify a backflip when they see one, but writing a precise programmatic definition is extremely difficult.

Their solution was to use human feedback. Instead of defining a reward function directly, they showed human evaluators pairs of short video clips and asked them to choose which one looked more like a backflip. From these comparisons, they **trained** a reward model that learned to score behaviours in a way that aligned with human judgments.

Ouyang et al. 2022 applies this same idea to language. Rather than assigning a numeric reward directly or imitating a fixed dataset of demonstrations, we learn from pairwise preferences. Given two model outputs, a human annotator indicates which one is better. The reward model is then trained to assign a higher score to preferred outputs than to less preferred ones.

In this setting, learning is driven by relative judgments—i.e. is option A preferred to option B—which provides a natural and scalable way to capture complex, nuanced human preferences in language.

https://arxiv.org/abs/1706.03741 - Deep Reinforcement Learning from Human Feedback (Christiano et al. 2017)

https://arxiv.org/abs/2203.02155 - Training language models to follow instructions with human feedback (Ouyang et al. 2022)

https://arxiv.org/abs/1712.01815 - Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm (Silver et al. 2017)


# Excercise: The Bradley Terry Model - How we go about this in practice?

In this section we will examine the bradley terry model, a common approach for learning reward models from paired feedback. We will first analyse a simple trivial example then apply this idea to a small LLM. **The first part of this practical should be done without AI Assistance.**

## Bradley–Terry Model

The Bradley–Terry model is a probabilistic model for learning preferences from
pairwise comparisons. It is commonly used when we observe data of the form
“item A is preferred to item B” and want to infer a latent score for each item.
This makes it a natural choice for training reward models in Reinforcement
Learning from Human Feedback (RLHF).

We define a question as $x$ and model response as $y$, each item $(x,y)$ is assumed to have a true unknown real-valued score $s(x,y)$. Given two items $(x,y_i), (x,y_j)$ the probability that item $i$ is preferred to $j$ is given as
$$
P(y_i \succ y_j | x) = \frac{\exp(s(x,y_i))}{\exp(s(x,y_i)) + \exp(s(x,y_j))}
$$


### Theorey Check

**Q1. Show that this can be re-written in terms of the score differences i.e. s_i - s_j**:


**Answer**:

Starting from the Bradley-Terry probability,
$$
P(y_i \succ y_j \mid x)
= \frac{\exp(s(x,y_i))}{\exp(s(x,y_i)) + \exp(s(x,y_j))}.
$$
Divide the numerator and denominator by $\exp(s(x,y_i))$:
$$
P(y_i \succ y_j \mid x)
= \frac{1}{1 + \exp\big(s(x,y_j) - s(x,y_i)\big)}.
$$
If we define the score difference
$$
\Delta s_{ij} = s(x,y_i) - s(x,y_j),
$$
then this becomes
$$
P(y_i \succ y_j \mid x)
= \frac{1}{1 + \exp(-\Delta s_{ij})}
= \sigma\big(s(x,y_i) - s(x,y_j)\big),
$$
where $\sigma(\cdot)$ is the logistic sigmoid. So the probability depends only on the score
_difference_ $s(x,y_i) - s(x,y_j)$, not on the absolute values of the scores.


**Q2. Given a dataset of comparisons $\{x_k, y_{k}, y'_{k} \}_{k=1}^K$ where $y_k$ is always preferred to $y'_{k}$, we create a parameterised score function $s_{\theta}(x,y)$ and learn it by maximizing the log-likelihood over the dataset using gradient based optimization methods.
 Write the log-likelihood function over the dataset**:

**Answer**:

Assuming the $K$ comparisons are conditionally independent given $\theta$, the likelihood factorises as
$$
\mathcal{L}(\theta) = \prod_{k=1}^{K} P_\theta\big(y_k \succ y'_k \mid x_k\big).
$$
Taking logs gives the dataset log-likelihood:
$$
\log \mathcal{L}(\theta)
= \sum_{k=1}^{K} \log P_\theta\big(y_k \succ y'_k \mid x_k\big)
= \sum_{k=1}^{K} \log \sigma\big(s_\theta(x_k,y_k) - s_\theta(x_k,y'_k)\big).
$$
Using $\log \sigma(z) = -\log(1 + e^{-z})$, we can also write this as
$$
\log \mathcal{L}(\theta)
= -\sum_{k=1}^{K} \log\Big(1 + \exp\big(-(s_\theta(x_k,y_k) - s_\theta(x_k,y'_k))\big)\Big).
$$
Training the reward model means maximising this log-likelihood, or equivalently minimising its negative.


**Q3. This can be thought of a binary classification problem where if $i \succ j$ the binary reward is $1$ and $0$ otherwise, think about how one might construct a similar likelihood objective over a ranking of K preferences:**
*Clue: Have a go at doing this yourself - think of it as a research problem - and then look at the appendix of the DPO paper which discusses the  Plackett-Luce model. Did you come up with something similar?*

**Answer**:

For a full ranking of $K$ responses,
$$
y_1 \succ y_2 \succ \cdots \succ y_K,
$$
a natural extension is to model the ranking as a sequence of "pick the best remaining item" decisions.
At stage $k$, the remaining set is $\mathcal{R}_k = \{y_k, y_{k+1}, \dots, y_K\}$, and we model the
probability that $y_k$ is chosen from that set with a softmax:
$$
P\big(y_k \text{ is top of } \mathcal{R}_k \mid x\big)
= \frac{\exp(s_\theta(x,y_k))}{\sum_{j=k}^{K} \exp(s_\theta(x,y_j))}.
$$
Multiplying these choices across the ranking gives the Plackett--Luce likelihood:
$$
P\big(y_1 \succ y_2 \succ \cdots \succ y_K \mid x\big)
= \prod_{k=1}^{K-1}
\frac{\exp(s_\theta(x,y_k))}{\sum_{j=k}^{K} \exp(s_\theta(x,y_j))}.
$$
For a dataset of ranked examples, the corresponding log-likelihood is
$$
\log \mathcal{L}(\theta)
= \sum_{n=1}^{N} \sum_{k=1}^{K-1}
\left[
 s_\theta(x_n, y_{n,k}) - \log \sum_{j=k}^{K} \exp(s_\theta(x_n, y_{n,j}))
\right].
$$
This is a sum of cross-entropy terms over the successive ranking decisions, and Bradley--Terry is recovered as the special case $K=2$.


### Practical

We now consider a simple example. Let $y \in \mathbb{R}^2$. We define the true reward function as
$ s(y) = y_{(1)}^3 - y_{(2)} $, generate a set of random data points in the space of $y$,
and label them according to this true reward function.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# True reward function (unknown to the model)
def true_reward(x):
    # x shape: [N, 2] where columns are (x, y)
    return x[:, 0]**3 - x[:, 1]

# Generate random points in [-1, 1]^2
def generate_pairs(n_pairs=10000):
    x1 = 2 * torch.rand(n_pairs, 2) - 1
    x2 = 2 * torch.rand(n_pairs, 2) - 1

    r1 = true_reward(x1)
    r2 = true_reward(x2)

    # Preference label: 1 if x1 preferred over x2
    y = (r1 > r2).float()

    return x1, x2, y

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate a dataset of points:
x1, x2, s = generate_pairs(n_pairs=50)

# Create grid
x = np.linspace(-2, 2, 400)
y = np.linspace(-2, 2, 400)
X, Y = np.meshgrid(x, y)

# Define the true reward on the grid
Z = X**3 - Y

# Filled contour plot
plt.figure()
contour = plt.contourf(X, Y, Z)

# Zero-level curve
plt.contour(X, Y, Z, levels=[0])

plt.title("Preference Learning Decision Boundary")
plt.xlabel("$y_{(1)}$")
plt.ylabel("$y_{(2)}$")
plt.colorbar(contour, label="$s(y) = y_{(1)}^3 - y_{(2)}$")

plt.scatter(x1[:, 0], x1[:, 1], c=s)
plt.scatter(x2[:, 0], x2[:, 1], c=1 - s)
plt.show()


**Q4. Write a suitable neural network architecture in pytorch to learn this decision boundary from the generated pairwise data points**


In [ ]:
import torch
import torch.nn as nn

# A small MLP is sufficient for this 2D nonlinear reward function.
class RewardModel(nn.Module):
    """
    Maps a 2D input y to a scalar score s_theta(y).
    """
    def __init__(self, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


#

**Q5. Add the Bradley Terry Loss to the standard pytorch training loop.**

In [ ]:
from torch.nn import functional as F

def bradley_terry_loss(r1_hat, r2_hat, y):
    """
    Bradley-Terry negative log-likelihood.

    Args:
        r1_hat: [N] predicted scores s_theta(x1)
        r2_hat: [N] predicted scores s_theta(x2)
        y: [N] labels in {0, 1}, where y=1 means x1 is preferred to x2
    """
    logits = r1_hat - r2_hat
    return F.binary_cross_entropy_with_logits(logits, y)


In [ ]:
from tqdm.notebook import tqdm

# Hyperparameters
batch_size = 256
epochs = 50
lr = 1e-3

# Data
x1, x2, y = generate_pairs()
dataset = torch.utils.data.TensorDataset(x1, x2, y)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Model + optimizer
model = RewardModel()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Training
losses = list()
for epoch in tqdm(range(epochs), desc="Training Bradley–Terry Model"):
    total_loss = 0.0

    for xb1, xb2, yb in loader:
        r1_hat = model(xb1)
        r2_hat = model(xb2)

        loss = bradley_terry_loss(r1_hat, r2_hat, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        losses.append(loss.item())

    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d} | Loss: {total_loss / len(loader):.4f}")

plt.figure()
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("Training Loss")
plt.title("Bradley–Terry Training Loss")
plt.show()


To test the learnt reward function we plot a countour plot of our learnt function over the true reward and examine the difference.

**Q6. What do you notice about the learnt reward function? What has been learnt and what can't be learnt in this setup?**


**Answer**:

The learned reward model typically recovers the _shape_ of the true reward landscape quite well, so the contour lines and local ordering of points are similar to the true function. In other words, it learns which points should be ranked above or below others.

What it cannot identify is the _absolute level_ of the reward. The Bradley--Terry likelihood depends only on score differences,
$$
\sigma\big(s_\theta(y) - s_\theta(y')\big),
$$
so adding a constant $c$ to every predicted score leaves the preference probabilities unchanged:
$$
\sigma\big((s_\theta(y)+c) - (s_\theta(y')+c)\big)
= \sigma\big(s_\theta(y) - s_\theta(y')\big).
$$
So this setup can learn the ordering and geometry of the reward surface, but not its absolute offset.



**Q7 [Extra]. What sort of data does the BT model well and what sort of preferences does the BT model not effectively model? Clue: Think about three points with different preference setups...**

**Answer**:

The Bradley--Terry model works well when preferences can be explained by a single latent scalar utility function. That means there exists some score $s(\cdot)$ such that
$$
a \succ b \iff s(a) > s(b).
$$
In that setting, preferences are transitive, so if $a \succ b$ and $b \succ c$, we should also have $a \succ c$.

BT struggles when preferences are intransitive or cyclic, for example
$$
a \succ b, \quad b \succ c, \quad c \succ a.
$$
No single scalar score can represent all three comparisons at once. In those cases, a BT model tends to assign similar scores and predict probabilities near $1/2$, so it fits none of the pairwise preferences well.

It also struggles when the preference label depends on missing context, such as different annotators having systematically different notions of what makes a response good.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Create grid
x = np.linspace(-2, 2, 400)
y = np.linspace(-2, 2, 400)
X, Y = np.meshgrid(x, y)

# True reward
Z_true = X**3 - Y

# Learned reward
grid = torch.from_numpy(
    np.stack([X.ravel(), Y.ravel()], axis=1)
).float()

with torch.no_grad():
    Z_hat = model(grid).numpy().reshape(X.shape)

# Shared contour levels (CRITICAL)
levels = np.linspace(Z_true.min(), Z_true.max(), 12)

# Plot
plt.figure()

# True reward as filled contours
cf = plt.contourf(
    X, Y, Z_true,
    levels=levels
)

# Learned reward as line contours (same levels!)
plt.contour(
    X, Y, Z_hat,
    levels=levels
)

plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(cf, label="Reward value")
plt.title("True Reward (color) vs Learned Reward (contours)")
plt.show()


## Reward Hacking

So problem solved right? Not quite, this is a good approach to learning but has some flaws that are difficult to address. A classic issue with RL is despite providing a reward the model fails to learn the desired behaviour. There are a variety of different types of problems that can result in this, Lilian Weng describes some in her blog (linked below), in this exercise we'll briefly consider two common failure modes that are broadly called: Reward Hacking. The first failure mode is typically defined as specifying a reward which doesn't actually lead to the correct model behaviour. A famous example is the game coast runner, where training on the game score lead to the model exploiting a different mechanic of the game rather than finishing the course as quickly as possible, see the Open AI Blog for more details.

The second mode is more subtle, even when the reward function is technically correct and reward the correct behaviour things can still go wrong. Langosco et al. 2023 show that deploying reward models out of distribution - moving from one domain to another - can cause issues even when the reward function was initially correct.


**Q8. Look at the toy example we solved above what do these two failure modes look like in this setting?**

**Answer**:

There are two clean failure modes in this toy setting.

First, we could use the wrong reward proxy when generating or learning preferences. For example, if the true objective were $s(y)=y_{(1)}^3-y_{(2)}$ but we collected labels using a simpler proxy like $\tilde{s}(y)=y_{(1)}-y_{(2)}$, then the model would faithfully learn the wrong target. This is the toy version of reward misspecification.

Second, we could train on one distribution and deploy on another. Here, if we train only on points from $[-1,1]^2$ and then evaluate on points from $[-2,2]^2$, the model may extrapolate badly outside the training region. In that case the reward was fine on-distribution, but fails out-of-distribution because of poor generalisation.


**Q9. What do these failure modes look like in the language setting? What problems might we encounter with reward models during RL training?**

**Answer**:

The same two issues appear in language reward modelling.

For misspecification, the reward model may pick up spurious shortcuts in the preference data instead of the behaviour we really want. Common examples include rewarding verbosity because longer answers are often preferred, or rewarding sycophancy because agreeing with the user is sometimes rated as helpful.

For out-of-distribution failure, the model may generalise poorly when it sees prompts, styles, or failure cases that were not well covered in the preference dataset. For example, if all bad answers in the training set contain obvious swear words, the reward model may simply learn a superficial swear-word detector rather than a genuine notion of helpfulness or safety.


In [ ]:
# Look at the correlation of the points in distribution:
with torch.no_grad():

    # Generate new test dataset
    test, _, _ = generate_pairs(n_pairs=100)

    r_true = true_reward(test)
    r_hat = model(test)

    corr = torch.corrcoef(torch.stack([r_true, r_hat]))[0, 1]
    print("Correlation with true reward:", corr.item())


# Generate out-of-distribution test points outside the training range [-1, 1]^2
def generate_bad_data(n_pairs=100):
    return 4 * torch.rand(n_pairs, 2) - 2

with torch.no_grad():

    # Generate new OOD test dataset
    test = generate_bad_data()

    r_true = true_reward(test)
    r_hat = model(test)

    corr = torch.corrcoef(torch.stack([r_true, r_hat]))[0, 1]
    print("Correlation with true reward OOD:", corr.item())


## Additional Exercise: Full Reward Model Learning

**You may use AI to help with specific steps, read the returned code carefully as these APIs are commonly used in research and very useful to know in modern ML research**


1. Select a preference dataset from huggingface datasets (https://huggingface.co/datasets) some popular examples are the HH-RLHF dataset.  

2. Select a model that will form the base of your reward model. For ease you will want a small 1B or 2B model. Using the API replace the last layer with a single output head that will return the reward.

3. Setup a pytorch learning loop using the code you have implemented above, train the model to completion, **Q. We can't plot the true reward function, what standard machine learning approaches can you use to monitor training progress?**

4. Once completed test your reward model, this can involve looking at the reward of samples you have generated your self or plugging it into a full training pipeline (also available on huggingface)...



**Answer**:

This is more an open question for you to complete and experiment so there is no single standard answer. Anyway, if you are interested in getting feedback, you can always send your solution to robin.faro@unibas.ch and we can discuss it.
